First Step: Load the required packages

In [103]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [104]:
sns.set_style('whitegrid')

Preprocessing GDP-percapita: First Step: Load the required dataset

In [105]:
file_path = "../../static/data/EU/Eurostat/GDP-percapita.xlsx"
raw_gdp = pd.read_excel(file_path, sheet_name="Sheet 1", header=None)

# Erste Vorschau der Daten
print(raw_gdp.head())

                                                  0   \
0  Data extracted on 21/12/2024 19:53:08 from [ES...   
1                                          Dataset:    
2                                     Last updated:    
3                                                NaN   
4                                     Time frequency   

                               1       2    3   4    5   6    7   8    9   \
0                             NaN     NaN  NaN NaN  NaN NaN  NaN NaN  NaN   
1  Real GDP per capita [tipsna40]     NaN  NaN NaN  NaN NaN  NaN NaN  NaN   
2                20/12/2024 11:00     NaN  NaN NaN  NaN NaN  NaN NaN  NaN   
3                             NaN     NaN  NaN NaN  NaN NaN  NaN NaN  NaN   
4                             NaN  Annual  NaN NaN  NaN NaN  NaN NaN  NaN   

   ...  49   50  51   52  53   54  55   56  57   58  
0  ... NaN  NaN NaN  NaN NaN  NaN NaN  NaN NaN  NaN  
1  ... NaN  NaN NaN  NaN NaN  NaN NaN  NaN NaN  NaN  
2  ... NaN  NaN NaN  NaN NaN  NaN NaN 

d:\Studium\2024W\DDOP\Exercise 2\.venv\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Now we process the GDP-percapita dataset

Change Column Names

In [106]:
# Manuelle Spaltennamen-Zuweisung
columns = [
    "GEO", "1995", "1996", "1997", "1998", "1999", "2000", 
    "2001", "2002", "2003", "2004", "2005", "2006", "2007",
    "2008", "2009", "2010", "2011", "2012", "2013", "2014",
    "2015", "2016", "2017", "2018", "2019", "2020", "2021",
    "2022", "2023"
]

# Zusätzliche Platzhalter für Metadaten-Spalten (falls nötig)
columns += [f"Meta_{i}" for i in range(1, 59 - len(columns) + 1)]

# Spaltennamen zuweisen
raw_gdp.columns = columns
print(raw_gdp.columns)


Index(['GEO', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002',
       '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011',
       '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020',
       '2021', '2022', '2023', 'Meta_1', 'Meta_2', 'Meta_3', 'Meta_4',
       'Meta_5', 'Meta_6', 'Meta_7', 'Meta_8', 'Meta_9', 'Meta_10', 'Meta_11',
       'Meta_12', 'Meta_13', 'Meta_14', 'Meta_15', 'Meta_16', 'Meta_17',
       'Meta_18', 'Meta_19', 'Meta_20', 'Meta_21', 'Meta_22', 'Meta_23',
       'Meta_24', 'Meta_25', 'Meta_26', 'Meta_27', 'Meta_28', 'Meta_29'],
      dtype='object')


Then we clean up the values

In [107]:
# Entferne Metadaten-Zeilen
clean_gdp = raw_gdp.dropna(subset=["GEO"])
clean_gdp = clean_gdp[~clean_gdp["GEO"].str.contains("TIME|GEO|Dataset|Last updated|Unit|Source", na=False)]
clean_gdp = clean_gdp[~clean_gdp["GEO"].str.match(r"^\d+$", na=False)]

# Nur relevante Spalten behalten
clean_gdp = clean_gdp[["GEO"] + [str(year) for year in range(1995, 2024)]]

# Konvertiere numerische Spalten zu Float (für Berechnungen)
for col in clean_gdp.columns[1:]:
    clean_gdp[col] = pd.to_numeric(clean_gdp[col], errors="coerce")

# Vorschau der bereinigten Daten
print(clean_gdp.info())
print(clean_gdp.head())


<class 'pandas.core.frame.DataFrame'>
Index: 38 entries, 0 to 45
Data columns (total 30 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   GEO     38 non-null     object 
 1   1995    28 non-null     float64
 2   1996    0 non-null      float64
 3   1997    28 non-null     float64
 4   1998    0 non-null      float64
 5   1999    28 non-null     float64
 6   2000    0 non-null      float64
 7   2001    28 non-null     float64
 8   2002    0 non-null      float64
 9   2003    28 non-null     float64
 10  2004    0 non-null      float64
 11  2005    29 non-null     float64
 12  2006    0 non-null      float64
 13  2007    29 non-null     float64
 14  2008    0 non-null      float64
 15  2009    29 non-null     float64
 16  2010    0 non-null      float64
 17  2011    29 non-null     float64
 18  2012    0 non-null      float64
 19  2013    29 non-null     float64
 20  2014    0 non-null      float64
 21  2015    29 non-null     float64
 22  2016    0

Now we clean the metadata and save the new dataset

In [108]:
# Entferne irrelevante Zeilen anhand bekannter Metadatenkennzeichen
clean_gdp = clean_gdp[~clean_gdp["GEO"].str.contains("Time frequency|Dataset|Last updated|Unit|Source|National accounts", na=False, case=False)]

# Behalte nur relevante Spalten mit Jahreswerten und der GEO-Spalte
columns_to_keep = ["GEO"] + [str(year) for year in range(1995, 2024)]
clean_gdp = clean_gdp[columns_to_keep]

# Entferne Spalten mit komplett fehlenden Werten
clean_gdp = clean_gdp.dropna(axis=1, how="all")

# Konvertiere numerische Spalten zu float für Berechnungen
for col in clean_gdp.columns[1:]:
    clean_gdp[col] = pd.to_numeric(clean_gdp[col], errors="coerce")

# Vorschau der bereinigten Daten
print(clean_gdp.info())
print(clean_gdp.head())

# Speichere die bereinigten Daten
clean_gdp.to_excel(file_path+"GDP-percapita_cleaned-Q4_.xlsx", index=False)


print("Bereinigte Daten wurden erfolgreich gespeichert!")


<class 'pandas.core.frame.DataFrame'>
Index: 36 entries, 0 to 45
Data columns (total 16 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   GEO     36 non-null     object 
 1   1995    28 non-null     float64
 2   1997    28 non-null     float64
 3   1999    28 non-null     float64
 4   2001    28 non-null     float64
 5   2003    28 non-null     float64
 6   2005    29 non-null     float64
 7   2007    29 non-null     float64
 8   2009    29 non-null     float64
 9   2011    29 non-null     float64
 10  2013    29 non-null     float64
 11  2015    29 non-null     float64
 12  2017    29 non-null     float64
 13  2019    29 non-null     float64
 14  2021    29 non-null     float64
 15  2023    29 non-null     float64
dtypes: float64(15), object(1)
memory usage: 4.8+ KB
None
                                                  GEO     1995     1997  \
0   Data extracted on 21/12/2024 19:53:08 from [ES...      NaN      NaN   
10          European Union -